# Comprobaciones de volumen del lakehouse

Cuenta las filas de cada tabla Delta (bronze, silver y gold) y las desglosa por fecha,
por estacion y por franja horaria segun lo que tenga cada tabla.

Sirve para tres cosas del plan:

1. verificar que existen las tablas que el codigo dice construir (`gold_mobility_pressure`),
2. detectar tablas vacias o con nombre distinto al esperado (`silver_mentions` vs `silver_csd_mentions`),
3. sacar las cifras de la seccion 7.2 de la memoria (metricas de volumen).

Sin logica de negocio: solo lee con `storage.read_delta` y agrega.


In [1]:
"""Comprobaciones de volumen de las tablas Delta. Solo lectura y conteo."""

import os

os.environ.setdefault("ENV", "local")

from pyspark.errors import AnalysisException
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

from multitudcsd.config import FECHA_REFERENCIA, get_lakehouse_root, get_spark_session
from multitudcsd.storage import read_delta

# Cuantas filas se enseñan como maximo en cada desglose.
FILAS_A_MOSTRAR = 40

spark = get_spark_session("comprobaciones-volumen")
spark.sparkContext.setLogLevel("ERROR")

print(f"[comprobaciones] lakehouse: {get_lakehouse_root()}")
print(f"[comprobaciones] fecha de referencia: {FECHA_REFERENCIA}")

[comprobaciones] lakehouse: D:/05_MasterUCM/TFM/multitudcsd/data/lakehouse
[comprobaciones] fecha de referencia: 2026-09-05


## 1. Catalogo de tablas y columnas de desglose

Una entrada por tabla del documento `tablas_delta.csv`. Para cada una se indica que columna
usar como fecha, cual como estacion y cual como hora. Si una clave no aparece, esa tabla no
admite ese desglose.

`fecha` se pasa siempre por `to_date`, asi que vale igual para una columna `date`, un
`timestamp` o un texto ISO.

In [2]:
# capa + columnas por las que tiene sentido desglosar el conteo de cada tabla.
# Falta una clave => esa tabla no admite ese desglose (p.ej. Gold no guarda fecha:
# to-do es del dia de referencia y el grano es celda H3 x hora).
TABLAS = {
    # --- BRONZE: todas llevan ingest_date, que es la particion -------------------
    "bronze_gtfs_tripupdates": {"capa": "bronze", "fecha": "ingest_date"},
    "bronze_gtfs_static_stops": {"capa": "bronze", "fecha": "ingest_date", "estacion": "stop_id"},
    "bronze_gtfs_static_stop_times": {"capa": "bronze", "fecha": "ingest_date", "estacion": "stop_id"},
    "bronze_gtfs_static_trips": {"capa": "bronze", "fecha": "ingest_date"},
    "bronze_gtfs_static_routes": {"capa": "bronze", "fecha": "ingest_date"},
    "bronze_gtfs_static_calendar": {"capa": "bronze", "fecha": "ingest_date"},
    "bronze_gtfs_static_calendar_dates": {"capa": "bronze", "fecha": "ingest_date"},
    "bronze_nextbike_status": {"capa": "bronze", "fecha": "ingest_date", "estacion": "station_id"},
    "bronze_nextbike_station_information": {"capa": "bronze", "fecha": "ingest_date", "estacion": "station_id"},
    "bronze_viz_disruptions": {"capa": "bronze", "fecha": "ingest_date"},
    "bronze_csd_mentions": {"capa": "bronze", "fecha": "ingest_date"},

    # --- SILVER: no hay ingest_date, la fecha sale del evento --------------------
    "silver_bike_availability": {"capa": "silver", "fecha": "reading_ts", "estacion": "station_id"},
    "silver_transit_delays": {"capa": "silver", "fecha": "feed_ts", "estacion": "stop_id"},
    "silver_disruptions": {"capa": "silver", "fecha": "valid_from"},
    "silver_transit_supply": {"capa": "silver", "estacion": "station_id", "hora": "scheduled_hour"},
    "silver_csd_mentions": {"capa": "silver", "fecha": "event_ts", "hora": "hour_of_day"},
    # Nombre alternativo que hoy escribe bronze_to_silver.py. Una de las dos debe fallar:
    # si fallan las dos, la etapa 2 no ha llegado a Silver.
    "silver_mentions": {"capa": "silver", "fecha": "event_ts", "hora": "hour_of_day"},

    # --- GOLD: agregados del dia de referencia, sin columna de fecha -------------
    "gold_mobility_pressure": {"capa": "gold", "hora": "hour_of_day"},
    "gold_line_reliability": {"capa": "gold", "hora": "hour_of_day"},
    "gold_disruptions_by_cell": {"capa": "gold"},
    "gold_station_services": {"capa": "gold", "estacion": "station_id"},
    "gold_transit_capacity": {"capa": "gold", "hora": "scheduled_hour"},
    "gold_csd_activity": {"capa": "gold", "hora": "hour_of_day"},
    "gold_mobility_vs_activity": {"capa": "gold", "hora": "hour_of_day"},
    "gold_delay_predictions": {"capa": "gold", "hora": "hour_of_day"},
}

print(f"[comprobaciones] {len(TABLAS)} tablas en el catalogo")

[comprobaciones] 25 tablas en el catalogo


## 2. Funciones auxiliares

In [3]:
def load_table(nombre_tabla: str, capa: str) -> DataFrame:
    """Lee una tabla del lakehouse, o devuelve None si todavia no existe en disco.

    Delta lanza AnalysisException cuando la ruta no existe o no es una tabla Delta. Aqui
    eso no es un error: significa que ese paso del pipeline no se ha ejecutado aun.
    """
    try:
        return read_delta(spark, capa, nombre_tabla)
    except AnalysisException:
        return None


def count_rows(nombre_tabla: str, capa: str) -> int:
    """Devuelve el numero de filas de una tabla, o -1 si no existe."""
    df = load_table(nombre_tabla, capa)
    if df is None:
        return -1
    return df.count()

## 3. Conteo total por tabla

Primer bloque de resultados: una linea por tabla con su numero de filas. `NO EXISTE`
significa que la carpeta Delta no esta escrita.

In [4]:
def print_row_counts() -> dict:
    """Cuenta las filas de todas las tablas del catalogo y las imprime en una tabla."""
    conteos = {}

    print(f"{'CAPA':<7} {'TABLA':<38} {'FILAS':>12}")
    print("-" * 60)
    for nombre_tabla, desglose in TABLAS.items():
        capa = desglose["capa"]
        filas = count_rows(nombre_tabla, capa)
        conteos[nombre_tabla] = filas
        texto_filas = "NO EXISTE" if filas < 0 else f"{filas:,}".replace(",", ".")
        print(f"{capa:<7} {nombre_tabla:<38} {texto_filas:>12}")

    existentes = [f for f in conteos.values() if f >= 0]
    vacias = [t for t, f in conteos.items() if f == 0]
    faltan = [t for t, f in conteos.items() if f < 0]

    print("-" * 60)
    print(f"[comprobaciones] tablas escritas: {len(existentes)} de {len(TABLAS)}")
    print(f"[comprobaciones] filas totales en el lakehouse: {sum(existentes):,}".replace(",", "."))
    if vacias:
        print(f"[comprobaciones] OJO, tablas existentes pero vacias: {', '.join(vacias)}")
    if faltan:
        print(f"[comprobaciones] tablas que faltan en disco: {', '.join(faltan)}")
    return conteos


conteos = print_row_counts()

CAPA    TABLA                                         FILAS
------------------------------------------------------------
bronze  bronze_gtfs_tripupdates                     246.429
bronze  bronze_gtfs_static_stops                      5.567
bronze  bronze_gtfs_static_stop_times             1.031.836
bronze  bronze_gtfs_static_trips                     87.305
bronze  bronze_gtfs_static_routes                       145
bronze  bronze_gtfs_static_calendar                   1.166
bronze  bronze_gtfs_static_calendar_dates            12.100
bronze  bronze_nextbike_status                       31.469
bronze  bronze_nextbike_station_information          17.833
bronze  bronze_viz_disruptions                        6.194
bronze  bronze_csd_mentions                           2.000
silver  silver_bike_availability                     14.685
silver  silver_transit_delays                       792.630
silver  silver_disruptions                              361
silver  silver_transit_supply          

## 4. Desglose por fecha, por estacion y por hora

Segundo bloque: para cada tabla que lo admite, filas agrupadas por la columna
correspondiente. El desglose por estacion se ordena de mas a menos filas y se recorta,
porque hay cientos de paradas.

In [5]:
def show_count_by_date(df: DataFrame, nombre_tabla: str, columna: str) -> None:
    """Filas por fecha. to_date sirve igual para date, timestamp o texto ISO."""
    resumen = (
        df.groupBy(F.to_date(F.col(columna)).alias("fecha"))
        .count()
        .orderBy("fecha")
    )
    print(f"  filas por fecha ({columna}):")
    resumen.show(FILAS_A_MOSTRAR, truncate=False)


def show_count_by_station(df: DataFrame, nombre_tabla: str, columna: str) -> None:
    """Filas por estacion o parada, de mas a menos, recortado a FILAS_A_MOSTRAR."""
    resumen = df.groupBy(F.col(columna).alias("estacion")).count().orderBy(F.desc("count"))
    num_estaciones = resumen.count()
    print(f"  filas por estacion ({columna}): {num_estaciones} estaciones distintas")
    resumen.show(FILAS_A_MOSTRAR, truncate=False)


def show_count_by_hour(df: DataFrame, nombre_tabla: str, columna: str) -> None:
    """Filas por franja horaria. En Gold es el unico eje temporal que existe."""
    resumen = df.groupBy(F.col(columna).alias("hora")).count().orderBy("hora")
    print(f"  filas por hora ({columna}):")
    resumen.show(24, truncate=False)


def print_breakdowns() -> None:
    """Recorre el catalogo y saca los desgloses que admite cada tabla."""
    for nombre_tabla, desglose in TABLAS.items():
        capa = desglose["capa"]
        df = load_table(nombre_tabla, capa)
        if df is None:
            continue

        print("=" * 70)
        print(f"[{capa}] {nombre_tabla}: {df.count():,} filas".replace(",", "."))

        if "fecha" in desglose:
            show_count_by_date(df, nombre_tabla, desglose["fecha"])
        if "estacion" in desglose:
            show_count_by_station(df, nombre_tabla, desglose["estacion"])
        if "hora" in desglose:
            show_count_by_hour(df, nombre_tabla, desglose["hora"])
        if not any(clave in desglose for clave in ("fecha", "estacion", "hora")):
            print("  sin desglose: el grano es celda H3, no tiene fecha ni estacion")


print_breakdowns()

[bronze] bronze_gtfs_tripupdates: 246.429 filas
  filas por fecha (ingest_date):
+----------+------+
|fecha     |count |
+----------+------+
|2026-08-23|6321  |
|2026-08-27|6264  |
|2026-09-02|5921  |
|2026-09-04|74195 |
|2026-09-08|153728|
+----------+------+

[bronze] bronze_gtfs_static_stops: 5.567 filas
  filas por fecha (ingest_date):
+----------+-----+
|fecha     |count|
+----------+-----+
|2026-09-06|5567 |
+----------+-----+

  filas por estacion (stop_id): 5567 estaciones distintas
+----------------------+-----+
|estacion              |count|
+----------------------+-----+
|de:11000:900100544::2 |1    |
|de:11000:900100530::1 |1    |
|de:11000:900100530::2 |1    |
|de:11000:900100537::1 |1    |
|de:11000:900100504::1 |1    |
|de:11000:900100526::2 |1    |
|de:11000:900100043::02|1    |
|de:11000:900110027::01|1    |
|de:11000:900100040::1 |1    |
|de:11000:900100009::10|1    |
|de:11000:900110520::2 |1    |
|de:11000:900110021::2 |1    |
|de:11000:900100503::3 |1    |
|de:1100

## 5. Cierre

Parar la sesion al terminar para liberar el puerto de la UI de Spark en local. En
Databricks la sesion es del cluster: no la pares alli.

In [6]:
if os.getenv("ENV", "local") == "local":
    spark.stop()
    print("[comprobaciones] sesion de Spark parada")

[comprobaciones] sesion de Spark parada
